# 🧠 CrewAI Hierarchical Agents Tutorial

**Goal:** Build a **manager → analysts** multi-agent workflow that:
- Scrapes the web (via **Firecrawl API** if available, else a built‑in fallback scraper)
- Ingests a **local PDF** into a vector DB (**FAISS** or **ChromaDB**)
- Runs **RAG** (Retrieval-Augmented Generation) to answer questions
- Uses **CrewAI** in a **hierarchical** setup where a manager delegates to specialized agents

> This is built for **beginners** and meant to run on **Colab**.

## Contents
1. Install dependencies (v1-compatible LangChain + CrewAI)  
2. Keys via Colab Secrets (OpenAI, optional Firecrawl)  
3. Choose vector store: **FAISS** or **ChromaDB**  
4. Load & chunk a PDF (your local report)  
5. Build embeddings + index (local, no Pinecone)  
6. Define RAG chain with citations  
7. Tools: `web_scrape` (Firecrawl-or-fallback), `rag_search`, `chunk_index`  
8. Agents: **Manager**, **Web Research Analyst**, **Data/RAG Analyst**, **Writer**  
9. Setup **hierarchical** Crew and run end-to-end  
10. Tips & extensions

## 1) Install dependencies

In [ ]:
# Keep versions compatible with LangChain v1 and community vectorstores
%pip -q install -U   "langchain==1.*"   "langchain-core==1.*"   "langchain-text-splitters==1.*"   "langchain-community==0.4.*"   "langchain-openai>=0.3.0"   "chromadb>=0.5.5"   faiss-cpu   pypdf tiktoken   crewai crewai-tools   beautifulsoup4 readability-lxml requests

%pip -q install firecrawl-py

## 2) Keys from Colab Secrets (only)
Set these in **Colab → Settings → Secrets** before running:
- `OPENAI_API_KEY` (**required**)
- `FIRECRAWL_API_KEY` (**optional**, only if you want to use Firecrawl API)

In [ ]:
import os
from google.colab import userdata

OPENAI_API_KEY    = userdata.get('OPENAI_API_KEY')
FIRECRAWL_API_KEY = userdata.get('FIRECRAWL_API_KEY')

assert OPENAI_API_KEY, "Missing OPENAI_API_KEY in Colab Secrets"

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if FIRECRAWL_API_KEY:
    os.environ["FIRECRAWL_API_KEY"] = FIRECRAWL_API_KEY

print("Keys loaded")

## 3) Choose your local vector store
You can pick **FAISS** (fast, in-memory) or **ChromaDB** (simple local persistence).

In [ ]:
USE_FAISS = True   # toggle button for ChromaDB and FAISS

# small helper to print what we use
print("Vector store:", "FAISS" if USE_FAISS else "ChromaDB")

## 4) Load & chunk your PDF
Upload your quarterly report PDF to Google Colab

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = "3D_Systems_2025_Q2_Report.pdf"

loader = PyPDFLoader(str(pdf_path))
docs = loader.load()  # list of Document (per page)

splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
splits = splitter.split_documents(docs)

print(f"Pages: {len(docs)}, Chunks: {len(splits)}")
print("Preview:", splits[0].page_content[:200].replace("\n"," "))

## 5) Embeddings + Index (FAISS or ChromaDB)
We'll use **OpenAI embeddings**. Make sure your **chunk size** isn't too tiny.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS, Chroma
import tempfile

emb = OpenAIEmbeddings(model="text-embedding-3-small")  # 1536 dims, cost-effective

if USE_FAISS:
    texts = [d.page_content for d in splits]
    metadatas = [d.metadata for d in splits]
    vectorstore = FAISS.from_texts(texts=texts, embedding=emb, metadatas=metadatas)
else:
    # persistent Chroma in a temp dir (you can set your own path)
    chroma_dir = tempfile.mkdtemp(prefix="chroma_")
    vectorstore = Chroma.from_documents(splits, embedding_function=emb, persist_directory=chroma_dir)
    vectorstore.persist()

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("✅ Vector store ready:", type(vectorstore).__name__)

## 6) RAG chain (with simple citations)
We'll fetch top-k chunks, render short snippets with page numbers, and ask the LLM to answer **only** from context.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI

def format_docs(docs):
    lines = []
    for i, d in enumerate(docs, 1):
        page = d.metadata.get("page", "?")
        snippet = d.page_content[:350].replace("\n"," ")
        lines.append(f"[{i}] (p.{page}) {snippet} ...")
    return "\n".join(lines)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer strictly from the provided context. If unknown, say you don't know."),
    ("human", "Question: {question}\n\nContext:\n{context}\n\nReturn a concise answer with [#] citations.")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever | RunnableLambda(format_docs),
     "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# quick smoke test
print(rag_chain.invoke("What were the main revenue drivers this quarter?"))

## 7) Tools for agents
We'll define:
- `web_scrape(url)` — uses **Firecrawl API** if `FIRECRAWL_API_KEY` provided, else falls back to a basic scraper.
- `rag_search(question)` — runs our RAG chain.
- `chunk_index(text)` — lets an agent add adhoc text into the vector DB (useful for scraped content).

> To keep it beginner-friendly, we use plain Python functions and register them in CrewAI tasks.

In [ ]:
import os, requests
from bs4 import BeautifulSoup
from readability import Document as ReadabilityDoc

def web_scrape(url: str) -> str:
    """Scrape and return main readable text from a URL.
    If FIRECRAWL_API_KEY is available, tries Firecrawl API.
    Otherwise falls back to requests + readability + bs4.
    """
    api_key = os.getenv("FIRECRAWL_API_KEY", "").strip()
    if api_key:
        try:
            # Generic Firecrawl scrape pattern
            resp = requests.post(
                "https://api.firecrawl.dev/v1/scrape",
                headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
                json={"url": url},
                timeout=60
            )
            resp.raise_for_status()
            data = resp.json()
            # try to pull content field; adjust if Firecrawl response differs
            content = data.get("content") or data.get("markdown") or data.get("text") or str(data)
            return str(content)[:8000]  # keep it bounded
        except Exception as e:
            print("[firecrawl fallback]", e)

    # Fallback scraper
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    doc = ReadabilityDoc(r.text)
    html = doc.summary()
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text[:8000]

def rag_search(question: str) -> str:
    """Answer from local vector store using RAG chain."""
    return rag_chain.invoke(question)

def chunk_index(text: str) -> str:
    """Add new text (e.g., scraped) into the vector DB to expand context."""
    if not text or not text.strip():
        return "No text provided."
    if isinstance(vectorstore, FAISS):
        new_vs = FAISS.from_texts([text], embedding=emb, metadatas=[{"source":"web"}])
        vectorstore.merge_from(new_vs)
        return "Added to FAISS index."
    elif isinstance(vectorstore, Chroma):
        vectorstore.add_texts([text], metadatas=[{"source":"web"}])
        vectorstore.persist()
        return "Added to Chroma index."
    else:
        return "Unknown vectorstore."

## 8) Define CrewAI Agents (Hierarchical)
We'll set up:
- **Manager** (has *no tools*, decides plan & delegates)
- **Web Research Analyst** (can call `web_scrape` and `chunk_index`)
- **RAG Analyst** (can call `rag_search` and `chunk_index`)
- **Writer** (no tools; composes final answer)

We'll run the crew with `Process.hierarchical` so the **Manager** directs the flow.

In [ ]:
from crewai import Agent, Task, Crew, Process

# We'll pass tools as callables in the Task descriptions (simple beginner style).
# For more advanced tool wiring, see crewai-tools package patterns.

manager = Agent(
    role="Project Manager",
    goal="Coordinate analysts to produce a grounded, cited answer to the user's question.",
    backstory="You are organized and ensure each specialist does their part.",
    verbose=True,
    allow_delegation=True,
)

web_analyst = Agent(
    role="Web Research Analyst",
    goal="Find reliable, relevant external information and add it to the knowledge base as needed.",
    backstory="You are great at finding and extracting the main ideas from the web.",
    verbose=True,
    allow_delegation=False,
)

rag_analyst = Agent(
    role="Data/RAG Analyst",
    goal="Use the local vector database to extract grounded facts and answer sub-questions.",
    backstory="You only use local RAG results and never hallucinate numbers.",
    verbose=True,
    allow_delegation=False,
)

writer = Agent(
    role="Writer",
    goal="Compose a concise, beginner-friendly final answer with [#] citations.",
    backstory="You write crisply and clearly for newcomers.",
    verbose=True,
    allow_delegation=False,
)

# Task templates
t_plan = Task(
    description=(
        "Plan how to answer the user question: '{question}'. "
        "Decide whether web research is needed (only if the PDF might be outdated or ambiguous). "
        "Assign sub-tasks to the web analyst and the RAG analyst. "
        "Request the writer to produce a final answer with [#] citations."
    ),
    expected_output="A short plan with roles and steps.",
    agent=manager,
)

t_web = Task(
    description=(
        "If web context is needed for '{question}', use web_scrape(url) on 1-2 credible sources. "
        "Summarize key points and then call chunk_index(summary) to add to the local DB. "
        "If not needed, explain briefly why and skip."
    ),
    expected_output="2-4 bullets of external findings and confirmation of indexing (if used).",
    agent=web_analyst,
)

t_rag = Task(
    description=(
        "Use rag_search(sub_question) to retrieve grounded facts from the local DB for '{question}'. "
        "Provide short, cited bullets referencing [#] from context."
    ),
    expected_output="3-5 bullets with [#] citations from local RAG.",
    agent=rag_analyst,
)

t_write = Task(
    description=(
        "Combine the analysts' outputs to produce a final answer to '{question}'. "
        "Keep it beginner-friendly (5-7 sentences) and include [#] citations where appropriate."
    ),
    expected_output="A cohesive, concise final answer with [#] citations.",
    agent=writer,
)

crew = Crew(
    agents=[manager, web_analyst, rag_analyst, writer],
    tasks=[t_plan, t_web, t_rag, t_write],
    process=Process.hierarchical,   # manager coordinates
    manager_agent=manager,
    verbose=True,
)
print("Crew ready.")

## 9) Run the hierarchical crew
Try a question about your uploaded report. If you want to avoid live web calls, phrase the question so RAG is sufficient.

In [ ]:
# Helper glue so analysts can call our Python functions when needed.
# For beginners, we keep it simple: we'll "simulate" tool usage by letting
# the notebook cell do the calls and feed results back into the crew via context.
#
# In production, you'd wrap tools with CrewAI tool adapters; for teaching,
# showing the function signatures and explicit calls is clearer.

def run_with_tools(user_question: str):
    # Step 1: Manager creates a plan
    plan = crew.kickoff(inputs={"question": user_question})
    print("\n=== PLAN ===\n", plan)

    # Minimal heuristic: if the user mentions "latest" or a URL, do a web scrape demo.
    need_web = any(k in user_question.lower() for k in ["latest", "news", "update", "guidance", "forecast", "outlook", "http"])

    web_notes = ""
    if need_web:
        try:
            url = "https://www.sec.gov/edgar/search/"  # placeholder; replace with a relevant URL if you want
            print("\n[web_scrape] scraping:", url)
            content = web_scrape(url)
            print(f"[web_scrape] got {len(content)} chars; indexing snippet...")
            web_notes = content[:1500]
            print(chunk_index(web_notes))
        except Exception as e:
            print("[web_scrape error]", e)

    # Step 2: RAG analyst pulls facts
    rag_notes = rag_search(user_question)
    print("\n=== RAG NOTES ===\n", rag_notes)

    # Step 3: Writer composes (simulate by calling LLM directly with combined context)
    from langchain_core.prompts import ChatPromptTemplate
    final_prompt = ChatPromptTemplate.from_messages([
        ("system", "Write a beginner-friendly answer. Cite as [#] where appropriate."),
        ("human", "Question: {q}\n\nRAG notes:\n{r}\n\nWeb notes (optional):\n{w}\n\nFinal answer:")
    ])
    llm_local = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    final_chain = final_prompt | llm_local | StrOutputParser()
    final_answer = final_chain.invoke({"q": user_question, "r": rag_notes, "w": web_notes})
    return final_answer

demo_q = input("please type in your question")
final = run_with_tools(demo_q)
print("\n=== FINAL ANSWER ===\n", final)

## 10) Tips & Extensions
- **Swap vector store**: set `USE_FAISS=False` to use **ChromaDB** (persists to a folder).  
- **Firecrawl**: set `FIRECRAWL_API_KEY` in Colab Secrets to use the Firecrawl API; otherwise the fallback scraper is used.  
- **Deeper tool wiring**: explore `crewai-tools` to formally register tools callable by agents without manual glue.  
- **Better citations**: store chunk IDs and inject exact [#] references from `retriever` hits.  
- **Multi-document**: load multiple PDFs and store metadata (e.g., company, quarter, year), then filter retrieval by metadata.